# ETE · Workshop 1 · Atividade: 30 m contra 10 m

**Pergunta:** quanto detalhe perdemos quando o píxel passa de 10 m (Sentinel-2) para 30 m (Landsat)?

**Como usar este caderno no Jupyter**
1. Corra as células **por ordem**, da 1 à 7: clique na célula e prima `Shift + Enter`.
2. Só a **Célula 4** se edita (a zona do recorte).
3. Coloque o ficheiro `incomati_seca.npz` na **mesma pasta** deste caderno. Se não o fizer, a Célula 2 procura-o sozinha em Downloads e noutras pastas, e se não o encontrar descarrega a imagem (precisa de Internet).
4. Se algo correr mal: menu **Kernel → Restart Kernel and Run All Cells**.

## Célula 1 · Chamar as ferramentas
**O que vai acontecer:** o Python abre três ferramentas: `os` (procurar ficheiros), `numpy` (contas com tabelas de números) e `matplotlib` (desenhar). A linha `%matplotlib inline` faz com que as imagens apareçam dentro do caderno.

**Deve aparecer:** `Ferramentas prontas.`

In [ ]:
%matplotlib inline
# Célula 1 · chamar as ferramentas
import os
import numpy as np
import matplotlib.pyplot as plt

print("Ferramentas prontas.")

## Célula 2 · Garantir que a imagem da seca existe
**O que vai acontecer, por esta ordem:**
1. Se `verde_seca` já está na memória, usa-a.
2. Se não, **procura** o ficheiro `incomati_seca.npz` (pasta do caderno, Downloads, Desktop, Documents e restante pasta pessoal). Aceita cópias como `incomati_seca (1).npz` e escolhe a mais recente.
3. Se não o encontrar, **descarrega** a imagem da Internet e guarda o ficheiro.

**Deve aparecer:** `Ficheiro encontrado e carregado: ...` e `Tamanho da imagem (linhas, colunas): (2711, 1991)`.

In [ ]:
# Célula 2 · garantir que a imagem da seca (verde_seca) existe
# Ordem: 1) já está na memória  2) procura o ficheiro .npz  3) descarrega da Internet
import glob
import sys
import subprocess

# ---- PARTE EDITÁVEL (só usada se for preciso descarregar) ----
SECA_INICIO, SECA_FIM = "2025-07-01", "2025-08-31"   # datas da lista S1-09
MAX_NUVENS = 20
# ---------------------------------------------------------------

def procurar_ficheiro(padrao):
    """Procura o ficheiro nas pastas habituais e, se preciso, em todo o seu utilizador.
    Devolve o caminho do mais recente, ou None."""
    casa = os.path.expanduser("~")
    pastas = [os.getcwd(), os.path.join(casa, "Downloads"),
              os.path.join(casa, "Desktop"), os.path.join(casa, "Documents"), casa]
    encontrados = []
    for pasta in pastas:
        encontrados += glob.glob(os.path.join(pasta, padrao))
    if not encontrados:
        encontrados = glob.glob(os.path.join(casa, "**", padrao), recursive=True)
    return max(encontrados, key=os.path.getmtime) if encontrados else None


def descarregar_seca():
    """Descarrega a banda verde da seca (Sentinel-2) para a AOI do ETE."""
    def importar():
        from shapely.geometry import Polygon, mapping
        import planetary_computer as pc
        from pystac_client import Client
        from odc.stac import load
        return Polygon, mapping, pc, Client, load

    try:
        Polygon, mapping, pc, Client, load = importar()
    except ImportError:
        print("A instalar as bibliotecas (1 a 2 minutos)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client",
                        "planetary-computer", "odc-stac", "shapely"], check=False)
        try:
            Polygon, mapping, pc, Client, load = importar()
        except ImportError:
            raise FileNotFoundError(
                "Não consegui instalar as bibliotecas (sem Internet?). "
                "Coloque o ficheiro incomati_seca.npz na pasta " + os.getcwd()
            )

    geom = Polygon([
        (32.578193171411186, -25.667719151558273),
        (32.775965704920395, -25.667719151558273),
        (32.775965704920395, -25.91196845209879),
        (32.578193171411186, -25.91196845209879),
        (32.578193171411186, -25.667719151558273),
    ])
    bbox = list(geom.bounds)

    try:
        catalogo = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                               modifier=pc.sign_inplace)
        busca = catalogo.search(collections=["sentinel-2-l2a"], intersects=mapping(geom),
                                datetime=f"{SECA_INICIO}/{SECA_FIM}",
                                query={"eo:cloud_cover": {"lt": MAX_NUVENS}})
        itens = list(busca.items())
        if not itens:
            raise ValueError("Sem imagens nessas datas. Aumente o intervalo ou MAX_NUVENS.")

        por_dia = {}
        for i in itens:
            por_dia.setdefault(i.datetime.date(), []).append(i)
        dia = min(por_dia, key=lambda d: np.mean(
            [i.properties["eo:cloud_cover"] for i in por_dia[d]]))
        escolhidos = por_dia[dia]
        print("Data escolhida:", dia)

        ds = load(escolhidos, bands=["B03"], bbox=bbox, crs="EPSG:32736",
                  resolution=10, groupby="solar_day")
        baseline = float(escolhidos[0].properties.get("s2:processing_baseline", "0"))
        desloc = 1000 if baseline >= 4.0 else 0
        dn = ds["B03"].isel(time=0).values.astype("float32")
        r = (dn - desloc) / 10000
        r[dn == 0] = np.nan
        return np.clip(r, 0, 1)
    except ValueError:
        raise
    except Exception as erro:
        raise FileNotFoundError(
            "Não consegui descarregar a imagem (" + type(erro).__name__ + "). "
            "Verifique a Internet ou coloque o ficheiro incomati_seca.npz na pasta "
            + os.getcwd()
        )


if "verde_seca" in globals():
    print("verde_seca já está na memória.")
else:
    caminho = procurar_ficheiro("incomati_seca*.npz")
    if caminho:
        verde_seca = np.load(caminho)["verde"]
        print("Ficheiro encontrado e carregado:", caminho)
    else:
        print("Ficheiro não encontrado. A descarregar a imagem da Internet...")
        verde_seca = descarregar_seca()
        np.savez("incomati_seca.npz", verde=verde_seca)
        print("Guardado em:", os.path.join(os.getcwd(), "incomati_seca.npz"))

print("Tamanho da imagem (linhas, colunas):", verde_seca.shape)

## Célula 3 · Ver a imagem inteira
**O que vai acontecer:** aparece a banda verde de toda a AOI, em cinzento, com uma grelha e os números das **linhas** (esquerda) e das **colunas** (em baixo). Procure uma faixa comprida e sinuosa: é o rio Incomáti. Anote a linha e a coluna de um ponto do rio.

In [ ]:
# Célula 3 · ver a imagem inteira, com os números das linhas e colunas
vmin, vmax = np.nanpercentile(verde_seca, [1, 99])

plt.figure(figsize=(7, 9))
plt.imshow(verde_seca, cmap="gray", vmin=vmin, vmax=vmax)
plt.grid(True, alpha=0.4)
plt.xlabel("Coluna (x)")
plt.ylabel("Linha (y)")
plt.title("Banda verde, época seca · procure o rio")
plt.show()

## Célula 4 · Escolher a zona do recorte ✏️ (a única que se edita)
**O que vai acontecer:** diz-se ao Python onde cortar. A célula corrige os números para o recorte caber na imagem e para serem múltiplos de 3 (por isso um `1000` pode aparecer como `999`; é normal).

| Variável | Significado |
| --- | --- |
| `LINHA_INICIAL` | Linha onde começa o recorte (da Célula 3) |
| `COLUNA_INICIAL` | Coluna onde começa o recorte (da Célula 3) |
| `TAMANHO` | Lado do recorte em píxeis (300 = 3 km) |

**Dica:** o recorte parte do canto superior esquerdo. Se o rio ficar de lado, recue 100 a 150 píxeis nos dois números.

In [ ]:
# Célula 4 · escolher a zona do recorte (PARTE EDITÁVEL)
LINHA_INICIAL  = 1000     # troque pelo número da linha onde está o rio
COLUNA_INICIAL = 800      # troque pelo número da coluna onde está o rio
TAMANHO        = 300      # recorte de 300 x 300 píxeis (3 km x 3 km)

# Garantir que o recorte cabe dentro da imagem e que os números são múltiplos de 3
linhas, colunas = verde_seca.shape
l0 = min(LINHA_INICIAL,  max(linhas  - TAMANHO, 0))
c0 = min(COLUNA_INICIAL, max(colunas - TAMANHO, 0))
l0 = (l0 // 3) * 3
c0 = (c0 // 3) * 3
t  = (min(TAMANHO, linhas - l0, colunas - c0) // 3) * 3

print(f"Recorte: linhas {l0} a {l0 + t}, colunas {c0} a {c0 + t} ({t * 10 / 1000:.1f} km de lado)")

## Célula 5 · Simular o Landsat (30 m) e recortar
**O que vai acontecer:** cria-se a imagem "tipo Landsat" guardando **1 píxel em cada 3**, em linhas e em colunas. Cada píxel novo cobre 30 m × 30 m. Depois cortam-se as duas imagens na **mesma zona**.

**Deve aparecer:** `Imagem completa a 10 m: (2711, 1991)` e `Imagem completa a 30 m: (904, 664)`.

In [ ]:
# Célula 5 · simular o Landsat (30 m) e recortar a mesma zona nas duas imagens
verde_30m = verde_seca[::3, ::3]      # fica 1 píxel em cada 3, em linhas e colunas

recorte_10m = verde_seca[l0:l0 + t, c0:c0 + t]
recorte_30m = verde_30m[l0 // 3:(l0 + t) // 3, c0 // 3:(c0 + t) // 3]

print("Imagem completa a 10 m:", verde_seca.shape)
print("Imagem completa a 30 m:", verde_30m.shape)

## Célula 6 · Desenhar as duas imagens lado a lado
**O que vai acontecer:** aparecem duas imagens da mesma zona: à esquerda a de 10 m (Sentinel-2), nítida; à direita a de 30 m (tipo Landsat), "aos quadrados".

**Pergunta para os pares:** qual mostra melhor os campos pequenos e as margens do rio? Para mapear uma cheia de 2013, que satélite usamos? *(Landsat: o Sentinel-2 só existe desde 2015.)*

In [ ]:
# Célula 6 · desenhar as duas imagens lado a lado
vmax = np.nanpercentile(recorte_10m, 99)
vmin = np.nanpercentile(recorte_10m, 1)

fig, eixos = plt.subplots(1, 2, figsize=(12, 6))
eixos[0].imshow(recorte_10m, cmap="gray", vmin=vmin, vmax=vmax)
eixos[0].set_title(f"10 m (Sentinel-2) · {recorte_10m.shape[0]}x{recorte_10m.shape[1]} píxeis")
eixos[1].imshow(recorte_30m, cmap="gray", vmin=vmin, vmax=vmax)
eixos[1].set_title(f"30 m (tipo Landsat) · {recorte_30m.shape[0]}x{recorte_30m.shape[1]} píxeis")
for e in eixos:
    e.axis("off")
plt.suptitle("A mesma zona, com píxeis de tamanhos diferentes")
plt.tight_layout()
plt.show()

## Célula 7 · Contar os píxeis e comparar
**O que vai acontecer:** o Python conta os píxeis de cada imagem. Na mesma zona de 3 km × 3 km, o Sentinel-2 tem 90 000 píxeis e o tipo Landsat 10 000: **9 vezes mais**, porque 900 m² ÷ 100 m² = 9.

**Deve aparecer:** `O Sentinel-2 tem 9 vezes mais píxeis na mesma zona.`

In [ ]:
# Célula 7 · contar os píxeis e comparar as áreas
n10 = recorte_10m.size
n30 = recorte_30m.size
print(f"Zona de {t * 10 / 1000:.1f} km x {t * 10 / 1000:.1f} km")
print(f"Píxeis a 10 m: {n10}  (cada um cobre 100 m²)")
print(f"Píxeis a 30 m: {n30}  (cada um cobre 900 m²)")
print(f"O Sentinel-2 tem {n10 / n30:.0f} vezes mais píxeis na mesma zona.")

## Erros comuns

| Última linha do erro | O que significa | Solução |
| --- | --- | --- |
| `NameError: name 'os' is not defined` (ou `np`, `plt`) | Saltou a Célula 1 | Corra a Célula 1 e depois as seguintes |
| `NameError: name 'verde_seca' is not defined` | Saltou a Célula 2 | Corra a Célula 2 |
| `NameError: name 'l0' is not defined` | Saltou a Célula 4 | Corra a Célula 4 e depois a 5 |
| `FileNotFoundError: Não consegui descarregar...` | Sem ficheiro e sem Internet | Ponha `incomati_seca.npz` na pasta do caderno |
| `ValueError: Sem imagens nessas datas` | Intervalo de datas sem imagens limpas | Aumente o intervalo de datas na Célula 2 |
| Imagem sem rio | Zona do recorte errada | Volte à Célula 3 e ajuste a Célula 4 |

Se nada resultar: **Kernel → Restart Kernel and Run All Cells**.